## Import libraries

In [2]:

import pandas as pd
import numpy as np
import re
from datetime import datetime
from google_play_scraper import app, reviews, Sort
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))
from scripts.text_cleaner import clean_text
from scripts.preprocessing import normalize_dates, remove_duplicates, validate_ratings

## Web Scraping

In [3]:
DASHEN_APP_ID = 'com.cr2.amolelight'

# app metadata 
app_info = app(
    DASHEN_APP_ID,
    lang='en', 
    country='et'  
)

print("=" * 50)
print("DASHEN Bank App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

DASHEN Bank App Info
App Title   : Dashen Mobile
Current Score: 4.2850466
Total Ratings: 2,112
Total Reviews: 513
Installs     : 500,000+


## Collecting reviews

In [4]:
print(f"Scraping reviews for Dashen...")

result, continuation_token = reviews(
    DASHEN_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       
    count=500,              
    filter_score_with=None 
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for Dashen...
Collected 500 raw reviews


## Inspecting data

In [5]:
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 8f4344ba-7a07-4368-b42b-4243b4905751
  userName: Mohamed Fedilu
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjVzh-xPh1pcdqrwBSlZUy9-3Exdbr7Ns9OmlVLrPNqENH39uIk
  content: waw
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: None
  at: 2026-02-26 01:18:20
  replyContent: None
  repliedAt: None
  appVersion: None


## Extracting needed fields

In [6]:
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Dashen Bank',
        'source'   : 'Google Play'
    })

df = pd.DataFrame(raw_data)

print(f"Shape: {df.shape}")
df.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,8f4344ba-7a07-4368-b42b-4243b4905751,waw,5,2026-02-26 01:18:20,Dashen Bank,Google Play
1,c3cdc9ab-dee1-4869-bfbc-74885b5d839c,ተመ,5,2026-02-21 18:12:45,Dashen Bank,Google Play
2,a853e05f-76f0-4a69-86ff-b24bc4e1bbec,worst bank ever ... they will take your money ...,1,2026-01-19 02:44:07,Dashen Bank,Google Play
3,978ac99b-6982-4bbb-ba3e-0ba3c196cab4,Nice,5,2026-01-12 18:55:06,Dashen Bank,Google Play
4,457731e1-6cf2-4bdb-99da-e3788e58a967,Very disappointing application. it's getting w...,1,2025-10-31 19:25:05,Dashen Bank,Google Play


## Exploring the Raw Data

In [7]:
print(f"Total reviews collected: {len(df)}")
print(f"\nColumn dtypes:")
print(df.dtypes)

Total reviews collected: 500

Column dtypes:
review_id            object
review               object
rating                int64
date         datetime64[ns]
bank                 object
source               object
dtype: object


## Rating distribution

In [8]:
print("Rating distribution:")
rating_counts = df['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  323  ████████████████████████████████████████████████████████████████
  4 stars:   53  ██████████
  3 stars:   37  ███████
  2 stars:   17  ███
  1 stars:   70  ██████████████


## Checking date column

In [9]:
print("Sample date values (raw):")
print(df['date'].head(10).to_string())

print(f"\nDate dtype: {df['date'].dtype}")

Sample date values (raw):
0   2026-02-26 01:18:20
1   2026-02-21 18:12:45
2   2026-01-19 02:44:07
3   2026-01-12 18:55:06
4   2025-10-31 19:25:05
5   2025-10-28 15:35:23
6   2025-10-27 19:37:55
7   2025-10-21 00:31:58
8   2025-09-09 22:24:44
9   2025-08-24 19:32:32

Date dtype: datetime64[ns]


## Missing Values

In [10]:
print("Missing Values")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

for col in df.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "Nothing missing :)"
    print(f"  {col:<15}: {status}")

Missing Values
  review_id      : Nothing missing :)
  review         : Nothing missing :)
  rating         : Nothing missing :)
  date           : Nothing missing :)
  bank           : Nothing missing :)
  source         : Nothing missing :)


## Duplicate Reviews

In [11]:
print("Duplicates")

exact_dupes = df.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

id_dupes = df.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

empty_reviews = (df['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Duplicates
  Exact duplicate reviews : 95
  Duplicate review IDs    : 0
  Empty review texts      : 0


## Remove Duplicates

In [12]:
df = remove_duplicates(df)

Removed 0 duplicate reviews


## Date Format

In [13]:
print("Date Format")
print(f"  Current dtype: {df['date'].dtype}")
print(f"  Sample values: {df['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Date Format
  Current dtype: datetime64[ns]
  Sample values: 2026-02-26 01:18:20
  Target format: YYYY-MM-DD (string or date object)


## Normalize dates

In [14]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

df = normalize_dates(df)

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-02-26 01:18:20
1   2026-02-21 18:12:45
2   2026-01-19 02:44:07
dtype: datetime64[ns]

After normalization:
0    2026-02-26
1    2026-02-21
2    2026-01-19
dtype: object

Date range: 2022-07-16 to 2026-02-26


## Clean Review Text

In [ ]:
sample = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample)}")
print(f"After : {repr(clean_text(sample))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


## Rating Validation

In [16]:
df = validate_ratings(df)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Removed 0 invalid ratings
Remaining: 500 reviews
Rating dtype: int64


## Cleaned data

In [17]:
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,waw,5,2026-02-26,Dashen Bank,Google Play
1,ተመ,5,2026-02-21,Dashen Bank,Google Play
2,worst bank ever ... they will take your money ...,1,2026-01-19,Dashen Bank,Google Play
3,Nice,5,2026-01-12,Dashen Bank,Google Play
4,Very disappointing application. it's getting w...,1,2025-10-31,Dashen Bank,Google Play
5,"Banking made simple, smart, and safe.",5,2025-10-28,Dashen Bank,Google Play
6,It is the best of all i liked it i used it alm...,5,2025-10-27,Dashen Bank,Google Play
7,gngu,5,2025-10-21,Dashen Bank,Google Play
8,"The app is very good , but it does not tell th...",4,2025-09-09,Dashen Bank,Google Play
9,The best app and easy to use. The only draw ba...,3,2025-08-24,Dashen Bank,Google Play


## Save cleaned data

In [18]:
output_path = '../data/processed/dashen_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/processed/dashen_reviews_clean.csv
